In [1]:
import pynucastro as pyna
import numpy as np
from pathlib import Path

In [2]:
rates=pyna.rates.library.Library(libfile=r"Nuclear_data\decays\actual\Reaclib_18_9_20")
rates_exp=pyna.rates.library.Library(libfile=r"Nuclear_data\decays\actual\Reaclib_default_Experimental")

sources_label_default=[]
for r in rates.get_rates():
    if r.source['Label'] not in sources_label_default:
        sources_label_default.append(r.source['Label'])
        

sources_label_experimental=[]
for r in rates_exp.get_rates():
    if r.source['Label'] not in sources_label_experimental:
        sources_label_experimental.append(r.source['Label'])
        

sources_label_theoretical=[]
for rate in rates.get_rates():
    if rate.source['Label'] not in sources_label_experimental:
        if rate.source['Label']  not in sources_label_theoretical:
            sources_label_theoretical.append(rate.source['Label'] )
            
#encuentro las teóricas alpha
filter_alpha=pyna.RateFilter(products=['he4'],
                             exact=False,
                             max_reactants=1,
                             max_products=2,
                             filter_function=lambda r: r.source['Label'] not in sources_label_experimental and r.Q>0 and r.reactants[0].Z==r.products[1].Z+r.products[0].Z and r.reactants[0].A==r.products[1].A+r.products[0].A
                             )

rates_alpha=rates.filter(filter_alpha)

In [4]:
output_file=Path('Nuclear_data/decays/example_reaclib') 
with output_file.open('w')as fout:
    fout.write('2'+' '*73 + '\n')
    fout.write(' '*74 + '\n')
    fout.write(' '*74 + '\n')
    for rate in rates_alpha.get_rates():
        name_p=rate.reactants[0].short_spec_name
        name_d=rate.products[1].short_spec_name
        source=rate.source['Label']
        Q_value=rate.Q
        lam=rate.eval(1e9)
        fout.write(' '*(10-len(name_p))+name_p+'  he4'+' '*(5-len(name_d))+name_d+' '*23+source+' '*6+f'{Q_value:.5e}'+'          '+'\n')
        lamd=np.log(lam)
        if lamd>0:
            fout.write(' '+f'{lamd:.5e}'+' 0.000000e+00 0.000000e+00 0.000000e+00                       '+'\n')
        else:
            fout.write(f'{lamd:.5e}'+' 0.000000e+00 0.000000e+00 0.000000e+00                       '+'\n')
        fout.write(' 0.000000e+00 0.000000e+00 0.000000e+00                                   '+'\n')